# Path-Signature Volatility Forecasting Experiments

This notebook is intentionally **results-first**. It suppresses the long feature-generation logs and only displays compact experiment tables.

It runs the same experiment sequence for all five requested horizons:

- **5 min**
- **1 hour**
- **1 day**
- **7 day**
- **30 day**

The experiment sequence is:

1. **Depth 3, full path:** run HAR, Random Forest, Signature + Ridge, and Signature + XGBoost.
2. **Depth 3 vs depth 4:** compare the two signature models.
3. **No augmentation vs time vs lead-lag + time:** compare the two signature models.
4. **Path dimensions:** compare price, price + activity, and the full source-compatible path.

For Binance, the full path is **price + activity + flow**. Yahoo does not contain signed trade flow, so the daily Yahoo experiments use **price + activity + range**, matching the original daily specification.

> **Why only the two signature models in experiments 2–4?** HAR uses only HAR volatility features and Random Forest uses only the summary-statistic features, so changing signature depth, augmentation, or path dimensions does not change their inputs. Re-fitting them in every ablation would duplicate the same results and add a lot of runtime/output.

## 1. Imports and quiet output

In [9]:
from pathlib import Path
from dataclasses import replace
import logging
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

from forecast_data import (
    SPECS,
    build_dataset,
    binance_to_bars,
    chronological_split,
    load_or_download_data,
)

from models_and_metrics import (
    fit_and_evaluate_standard_models,
    fit_signature_ridge,
    fit_signature_xgb,
    evaluate_model,
)

# Keep notebook output focused on final result tables.
# Change WARNING -> INFO if you want detailed progress.
logging.getLogger().setLevel(logging.WARNING)
logging.getLogger("forecast_data").setLevel(logging.WARNING)
logging.getLogger("models_and_metrics").setLevel(logging.WARNING)

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

## 2. Configuration
Change dates and experiment settings here.

In [10]:
DATA_ROOT = Path("data")
OUTPUT_ROOT = Path("outputs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# =================================================
# DATA RANGES
# =================================================

# Binance is used for 5-minute and 1-hour forecasts.
BINANCE_SYMBOL = "BTCUSDT"
BINANCE_START = "2026-08-01"
BINANCE_END = "2026-08-31"       # inclusive

# Yahoo is used for 1-day, 7-day, and 30-day forecasts.
YAHOO_TICKER = "BTC-USD"
YAHOO_START = "2020-01-01"
YAHOO_END = "2026-09-16"         # inclusive

# Existing cached files are reused unless this is True.
FORCE_DOWNLOAD = False

# Final chronological holdout.
TEST_FRACTION = 0.20

# Main metric shown in the compact tables.
# Change to "MAE" or "QLIKE" if desired.
PRIMARY_METRIC = "RMSE"

# All requested horizons.
HORIZONS = ["5m", "1h", "1d", "7d", "30d"]

HORIZON_LABELS = {
    "5m": "5 min",
    "1h": "1 hour",
    "1d": "1 day",
    "7d": "7 day",
    "30d": "30 day",
}

# Keep XGBoost enabled to run all requested signature models.
INCLUDE_XGB = True

## 3. Helper functions

These helpers:

1. load each raw source only once,
2. cache each unique forecast-dataset specification, and
3. return compact result DataFrames instead of printing feature-generation details.

In [21]:
RAW_CACHE = {}
BARS_CACHE = {}
DATASET_CACHE = {}


def horizon_source(horizon):
    if horizon in {"5m", "1h"}:
        return {
            "source": "binance",
            "start": BINANCE_START,
            "end": BINANCE_END,
            "symbol": BINANCE_SYMBOL,
        }

    return {
        "source": "yahoo",
        "start": YAHOO_START,
        "end": YAHOO_END,
        "ticker": YAHOO_TICKER,
    }


def get_raw_data(horizon):
    cfg = horizon_source(horizon)
    source = cfg["source"]

    if source == "binance":
        cache_key = (
            "binance",
            BINANCE_SYMBOL,
            BINANCE_START,
            BINANCE_END,
        )
    else:
        cache_key = (
            "yahoo",
            YAHOO_TICKER,
            YAHOO_START,
            YAHOO_END,
        )

    if cache_key not in RAW_CACHE:
        RAW_CACHE[cache_key] = load_or_download_data(
            source=source,
            start=cfg["start"],
            end=cfg["end"],
            symbol=cfg.get("symbol", BINANCE_SYMBOL),
            ticker=cfg.get("ticker", YAHOO_TICKER),
            data_root=DATA_ROOT,
            force_download=FORCE_DOWNLOAD,
        )

    return RAW_CACHE[cache_key]


def get_bars(horizon):
    if horizon in BARS_CACHE:
        return BARS_CACHE[horizon]

    raw = get_raw_data(horizon)
    spec = SPECS[horizon]

    if horizon in {"5m", "1h"}:
        bars = binance_to_bars(raw, spec.bar_freq)
    else:
        bars = raw.copy()

    BARS_CACHE[horizon] = bars
    return bars


def spec_key(horizon, spec):
    return (
        horizon,
        spec.depth,
        spec.lead_lag,
        spec.time_aug,
        tuple(spec.dims),
        spec.path_points,
        spec.sample_stride,
        spec.variance_method,
    )


def get_dataset(horizon, spec=None):
    if spec is None:
        spec = SPECS[horizon]

    key = spec_key(horizon, spec)

    if key not in DATASET_CACHE:
        DATASET_CACHE[key] = build_dataset(
            get_bars(horizon),
            spec,
        )

    return DATASET_CACHE[key]


def signature_model_results(dataset):
    train_idx, test_idx = chronological_split(
        dataset,
        test_fraction=TEST_FRACTION,
    )

    y_train = dataset.y[train_idx]
    y_test = dataset.y[test_idx]

    results = {}

    ridge = fit_signature_ridge(
        dataset.X_sig[train_idx],
        y_train,
    )
    results["Signature Ridge"] = evaluate_model(
        "Signature Ridge",
        ridge,
        dataset.X_sig[test_idx],
        y_test,
    )

    if INCLUDE_XGB:
        xgb = fit_signature_xgb(
            dataset.X_sig[train_idx],
            y_train,
        )
        results["Signature XGBoost"] = evaluate_model(
            "Signature XGBoost",
            xgb,
            dataset.X_sig[test_idx],
            y_test,
        )

    return results


def result_rows(results, horizon, experiment, setting):
    rows = []

    for model_name, result in results.items():
        rows.append({
            "Horizon": HORIZON_LABELS[horizon],
            "HorizonKey": horizon,
            "Experiment": experiment,
            "Setting": setting,
            "Model": model_name,
            **result.metrics,
        })

    return rows


def ordered_results(df):
    out = df.copy()
    out["Horizon"] = pd.Categorical(
        out["Horizon"],
        categories=[HORIZON_LABELS[h] for h in HORIZONS],
        ordered=True,
    )
    return out.sort_values(["RMSE", "QLIKE"])


def show_baseline_table(df, metric=PRIMARY_METRIC):
    table = (
        df.pivot(
            index="Horizon",
            columns="Model",
            values=metric,
        )
        .reindex([HORIZON_LABELS[h] for h in HORIZONS])
    )

    display(
        table.style
        .format("{:.6g}")
        .highlight_min(axis=1, props="font-weight: bold;")
        .set_caption(f"Baseline — {metric} (bold = lowest within horizon)")
    )

    return table


def show_ablation_table(df, metric=PRIMARY_METRIC):
    table = df.pivot_table(
        index=["Horizon", "Model"],
        columns="Setting",
        values=metric,
        aggfunc="first",
    )

    horizon_order = [HORIZON_LABELS[h] for h in HORIZONS]
    order_map = {h: i for i, h in enumerate(horizon_order)}

    temp = table.reset_index()
    temp["_order"] = temp["Horizon"].map(order_map)
    temp = temp.sort_values(["_order", "Model"]).drop(columns="_order")
    table = temp.set_index(["Horizon", "Model"])

    display(
        table.style
        .format("{:.6g}")
        .highlight_min(axis=1, props="font-weight: bold;")
        .set_caption(f"{metric} comparison (bold = lowest across settings in each row)")
    )

    return table


def show_full_metrics(df):
    cols = ["Horizon", "Experiment", "Setting", "Model", "RMSE", "MAE", "QLIKE"]
    display(
        ordered_results(df)[cols]
        .style
        .format({
            "RMSE": "{:.6g}",
            "MAE": "{:.6g}",
            "QLIKE": "{:.6g}",
        })
    )

## 4. Experiment 1 — Depth 3, full path, all four models

This is the main baseline.

For each horizon:

- **HAR-RV** uses HAR volatility features.
- **Random Forest** uses summary statistics.
- **Signature Ridge** uses depth-3 signatures.
- **Signature XGBoost** uses depth-3 signatures.

The baseline signature path uses both **lead-lag and time augmentation**.

- 5 min / 1 hour: `price + activity + flow`
- 1 day / 7 day / 30 day: `price + activity + range`

In [12]:
baseline_rows = []
baseline_results = {}
baseline_datasets = {}

for horizon in HORIZONS:
    spec = replace(
        SPECS[horizon],
        depth=3,
        lead_lag=True,
        time_aug=True,
    )

    dataset = get_dataset(horizon, spec)
    train_idx, test_idx = chronological_split(
        dataset,
        test_fraction=TEST_FRACTION,
    )

    results = fit_and_evaluate_standard_models(
        dataset,
        train_idx,
        test_idx,
        include_xgb=INCLUDE_XGB,
    )

    baseline_datasets[horizon] = dataset
    baseline_results[horizon] = results

    baseline_rows.extend(
        result_rows(
            results,
            horizon=horizon,
            experiment="Baseline",
            setting="Depth 3 | full path | lead-lag + time",
        )
    )

baseline_df = ordered_results(pd.DataFrame(baseline_rows))
baseline_table = show_baseline_table(baseline_df)

/Users/mihikaghaisas/miniforge3/envs/test1/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=3.12705e-08): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/Users/mihikaghaisas/miniforge3/envs/test1/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=1.00253e-08): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/Users/mihikaghaisas/miniforge3/envs/test1/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=5.54345e-09): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/Users/mihikaghaisas/miniforge3/envs/test1/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=2.69754e-09): result may not be accurate.
  return linalg.solve(A, 

Model,HAR,Random Forest,Signature Ridge,Signature XGBoost
Horizon,,,,
5 min,0.000377043,0.000481073,0.000466614,0.000424682
1 hour,0.00124903,0.00268305,0.00189911,0.00151339
1 day,0.010062,0.011679,0.0095585,0.0100768
7 day,0.0199441,0.0317803,0.0191543,0.0236441
30 day,0.0416085,0.0595731,0.0399165,0.0457443


### Optional: inspect all baseline metrics

The compact table above shows only `PRIMARY_METRIC`. Run this cell only when you want RMSE, MAE, and QLIKE together.

In [ ]:
# Uncomment when needed:
# show_full_metrics(baseline_df)

## 5. Experiment 2 — Depth 3 vs depth 4

Everything except signature depth is held fixed:

- full source-compatible path dimensions
- lead-lag augmentation
- time augmentation

Only the two signature models are shown because HAR and Random Forest are unaffected by signature depth.

In [ ]:
# depth_rows = []

# for horizon in HORIZONS:
#     for depth in [3, 4]:
#         spec = replace(
#             SPECS[horizon],
#             depth=depth,
#             lead_lag=True,
#             time_aug=True,
#         )

#         dataset = get_dataset(horizon, spec)
#         results = signature_model_results(dataset)

#         depth_rows.extend(
#             result_rows(
#                 results,
#                 horizon=horizon,
#                 experiment="Depth",
#                 setting=f"Depth {depth}",
#             )
#         )

# depth_df = ordered_results(pd.DataFrame(depth_rows))
# depth_table = show_ablation_table(depth_df)

/Users/mihikaghaisas/miniforge3/envs/test1/lib/python3.10/site-packages/joblib/externals/loky/process_executor.py:700: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/Users/mihikaghaisas/miniforge3/envs/test1/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=3.12705e-08): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/Users/mihikaghaisas/miniforge3/envs/test1/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=1.00253e-08): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/Users/mihikaghaisas/miniforge3/envs/test1/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=5.54345e-09): result may not be accurate.
  ret

## 6. Experiment 3 — Augmentation

Depth is fixed at 3 and the full source-compatible dimensions are used.

The three paths are:

1. **None** — raw path only
2. **Time** — time augmentation only
3. **Lead-lag + time** — lead-lag transform followed by time augmentation

In [13]:
AUGMENTATIONS = {
    "None": {
        "lead_lag": False,
        "time_aug": False,
    },
    "Time": {
        "lead_lag": False,
        "time_aug": True,
    },
    "Lead-lag + time": {
        "lead_lag": True,
        "time_aug": True,
    },
}

augmentation_rows = []

for horizon in HORIZONS:
    for setting, kwargs in AUGMENTATIONS.items():
        spec = replace(
            SPECS[horizon],
            depth=3,
            lead_lag=kwargs["lead_lag"],
            time_aug=kwargs["time_aug"],
        )

        dataset = get_dataset(horizon, spec)
        results = signature_model_results(dataset)

        augmentation_rows.extend(
            result_rows(
                results,
                horizon=horizon,
                experiment="Augmentation",
                setting=setting,
            )
        )

augmentation_df = ordered_results(pd.DataFrame(augmentation_rows))
augmentation_table = show_ablation_table(augmentation_df)

/Users/mihikaghaisas/miniforge3/envs/test1/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=3.17606e-08): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/Users/mihikaghaisas/miniforge3/envs/test1/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=2.73173e-08): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/Users/mihikaghaisas/miniforge3/envs/test1/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=2.10737e-08): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/Users/mihikaghaisas/miniforge3/envs/test1/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=9.83867e-09): result may not be accurate.
  return linalg.solve(A, 

## 7. Experiment 4 — Path dimensions

Depth is fixed at 3 and **lead-lag + time augmentation** is used throughout.

### Binance: 5 min and 1 hour
- Price
- Price + activity
- Price + activity + flow

### Yahoo: 1 day, 7 day, 30 day
Yahoo OHLCV does **not** have signed buyer/seller trade flow. To keep the experiment valid with the existing Yahoo source, the third dimension is the daily range path from the original specification:

- Price
- Price + activity
- Price + activity + range

If you later switch the daily horizons to a source containing signed trade flow, change the Yahoo full-dimension tuple to `("price", "activity", "flow")`.

In [14]:
def dimension_variants(horizon):
    if horizon in {"5m", "1h"}:
        return {
            "Price": ("price",),
            "Price + activity": ("price", "activity"),
            "Price + activity + flow": ("price", "activity", "flow"),
        }

    return {
        "Price": ("price",),
        "Price + activity": ("price", "activity"),
        "Price + activity + range": ("price", "activity", "range"),
    }


dimension_rows = []

for horizon in HORIZONS:
    for setting, dims in dimension_variants(horizon).items():
        spec = replace(
            SPECS[horizon],
            depth=3,
            lead_lag=True,
            time_aug=True,
            dims=dims,
        )

        dataset = get_dataset(horizon, spec)
        results = signature_model_results(dataset)

        dimension_rows.extend(
            result_rows(
                results,
                horizon=horizon,
                experiment="Dimensions",
                setting=setting,
            )
        )

dimension_df = ordered_results(pd.DataFrame(dimension_rows))
dimension_table = show_ablation_table(dimension_df)

/Users/mihikaghaisas/miniforge3/envs/test1/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=3.74633e-08): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/Users/mihikaghaisas/miniforge3/envs/test1/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=2.91713e-08): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/Users/mihikaghaisas/miniforge3/envs/test1/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=7.39488e-09): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/Users/mihikaghaisas/miniforge3/envs/test1/lib/python3.10/site-packages/sklearn/linear_model/_ridge.py:213: LinAlgWarning: Ill-conditioned matrix (rcond=2.19683e-08): result may not be accurate.
  return linalg.solve(A, 

## 8. Combined results

This combines every result into one tidy DataFrame. The four compact experiment tables above are the main result views.

In [24]:
all_results = pd.concat(
    [
        baseline_df,
        #depth_df,
        augmentation_df,
        dimension_df,
    ],
    ignore_index=True,
)

all_results = ordered_results(all_results)

display(
    all_results[
        ["Horizon", "Experiment", "Setting", "Model", "RMSE", "MAE", "QLIKE"]
    ].head(20)
    .style
    .format({
        "RMSE": "{:.6g}",
        "MAE": "{:.6g}",
        "QLIKE": "{:.6g}",
    })
)

,Horizon,Experiment,Setting,Model,RMSE,MAE,QLIKE
0,5 min,Baseline,Depth 3 | full path | lead-lag + time,HAR,0.000377043,0.000232669,-13.2088
54,5 min,Dimensions,Price + activity,Signature XGBoost,0.000416018,0.000261631,-13.1257
53,5 min,Dimensions,Price,Signature XGBoost,0.000422925,0.000262938,-13.0916
3,5 min,Baseline,Depth 3 | full path | lead-lag + time,Signature XGBoost,0.000424682,0.000267869,-13.1273
23,5 min,Augmentation,Lead-lag + time,Signature XGBoost,0.000424682,0.000267869,-13.1273
55,5 min,Dimensions,Price + activity + flow,Signature XGBoost,0.000424682,0.000267869,-13.1273
51,5 min,Dimensions,Price + activity,Signature Ridge,0.000453377,0.000285192,-12.8423
2,5 min,Baseline,Depth 3 | full path | lead-lag + time,Signature Ridge,0.000466614,0.00029502,2000.09
20,5 min,Augmentation,Lead-lag + time,Signature Ridge,0.000466614,0.00029502,2000.09
52,5 min,Dimensions,Price + activity + flow,Signature Ridge,0.000466614,0.00029502,2000.09


In [25]:
df_min5 = all_results[all_results["Horizon"] == "5 min"]
df_min5.head(5)

,Horizon,HorizonKey,Experiment,Setting,Model,RMSE,MAE,QLIKE
0,5 min,5m,Baseline,Depth 3 | full path | lead-lag + time,HAR,0.000377,0.000233,-13.208760
54,5 min,5m,Dimensions,Price + activity,Signature XGBoost,0.000416,0.000262,-13.125722
53,5 min,5m,Dimensions,Price,Signature XGBoost,0.000423,0.000263,-13.091577
3,5 min,5m,Baseline,Depth 3 | full path | lead-lag + time,Signature XGBoost,0.000425,0.000268,-13.127328
23,5 min,5m,Augmentation,Lead-lag + time,Signature XGBoost,0.000425,0.000268,-13.127328


In [27]:
df_hr1 = all_results[all_results["Horizon"] == "1 hour"]
df_hr1.head(5)

,Horizon,HorizonKey,Experiment,Setting,Model,RMSE,MAE,QLIKE
4,1 hour,1h,Baseline,Depth 3 | full path | lead-lag + time,HAR,0.001249,0.000878,-10.338301
7,1 hour,1h,Baseline,Depth 3 | full path | lead-lag + time,Signature XGBoost,0.001513,0.001044,-10.179659
29,1 hour,1h,Augmentation,Lead-lag + time,Signature XGBoost,0.001513,0.001044,-10.179659
61,1 hour,1h,Dimensions,Price + activity + flow,Signature XGBoost,0.001513,0.001044,-10.179659
59,1 hour,1h,Dimensions,Price,Signature XGBoost,0.001579,0.001068,-10.168032


In [28]:
df_d1 = all_results[all_results["Horizon"] == "1 day"]
df_d1.head(5)

,Horizon,HorizonKey,Experiment,Setting,Model,RMSE,MAE,QLIKE
10,1 day,1d,Baseline,Depth 3 | full path | lead-lag + time,Signature Ridge,0.009559,0.006788,-6.707921
32,1 day,1d,Augmentation,Lead-lag + time,Signature Ridge,0.009559,0.006788,-6.707921
64,1 day,1d,Dimensions,Price + activity + range,Signature Ridge,0.009559,0.006788,-6.707921
34,1 day,1d,Augmentation,Time,Signature Ridge,0.009626,0.006912,-6.718954
63,1 day,1d,Dimensions,Price + activity,Signature Ridge,0.009667,0.007117,-6.721348


In [29]:
df_d7 = all_results[all_results["Horizon"] == "7 day"]
df_d7.head(5)

,Horizon,HorizonKey,Experiment,Setting,Model,RMSE,MAE,QLIKE
40,7 day,7d,Augmentation,Time,Signature Ridge,0.019101,0.014316,-4.698436
14,7 day,7d,Baseline,Depth 3 | full path | lead-lag + time,Signature Ridge,0.019154,0.014508,-4.698752
38,7 day,7d,Augmentation,Lead-lag + time,Signature Ridge,0.019154,0.014508,-4.698752
70,7 day,7d,Dimensions,Price + activity + range,Signature Ridge,0.019154,0.014508,-4.698752
39,7 day,7d,Augmentation,None,Signature Ridge,0.019399,0.014440,-4.681785


In [30]:
df_d30 = all_results[all_results["Horizon"] == "30 day"]
df_d30.head(5)

,Horizon,HorizonKey,Experiment,Setting,Model,RMSE,MAE,QLIKE
46,30 day,30d,Augmentation,Time,Signature Ridge,0.039053,0.034426,-3.159351
77,30 day,30d,Dimensions,Price,Signature XGBoost,0.039554,0.033635,-3.162874
18,30 day,30d,Baseline,Depth 3 | full path | lead-lag + time,Signature Ridge,0.039916,0.035345,-3.153945
44,30 day,30d,Augmentation,Lead-lag + time,Signature Ridge,0.039916,0.035345,-3.153945
76,30 day,30d,Dimensions,Price + activity + range,Signature Ridge,0.039916,0.035345,-3.153945


## 9. Save the result tables

This saves only the compact/tidy experiment results, not large feature arrays.

In [16]:
baseline_df.to_csv(
    OUTPUT_ROOT / "baseline_results.csv",
    index=False,
)

# depth_df.to_csv(
#     OUTPUT_ROOT / "depth_ablation_results.csv",
#     index=False,
# )

augmentation_df.to_csv(
    OUTPUT_ROOT / "augmentation_ablation_results.csv",
    index=False,
)

dimension_df.to_csv(
    OUTPUT_ROOT / "dimension_ablation_results.csv",
    index=False,
)

all_results.to_csv(
    OUTPUT_ROOT / "all_experiment_results.csv",
    index=False,
)

print("Saved compact result CSVs to:", OUTPUT_ROOT.resolve())

Saved compact result CSVs to: /Users/mihikaghaisas/Desktop/CMU/MLCapstone/Mihika/outputs


## 10. Optional — switch the displayed metric

Every model run already stores RMSE, MAE, and QLIKE. You do **not** need to refit anything to change the displayed metric.

In [ ]:
# Examples — no model refitting required:
#
# show_baseline_table(baseline_df, metric="MAE")
# show_ablation_table(depth_df, metric="MAE")
# show_ablation_table(augmentation_df, metric="QLIKE")
# show_ablation_table(dimension_df, metric="QLIKE")